# 🎙️ 日本語文字起こし(faster-whisper → SRT)

[動画圧縮ツール](https://mameogalaxy.github.io/dougaayuku/)で抽出した音声(.m4a など)を、faster-whisper で日本語文字起こしして、**タイムスタンプ付きSRTファイル**をダウンロードします。

## 使い方
1. **GPUを有効にする(推奨)**: 上のメニュー「ランタイム」→「ランタイムのタイプを変更」→ ハードウェア アクセラレータで「**T4 GPU**」を選んで保存
2. 下の **セル1** を実行(▶ボタン)— 初回は1〜2分かかります
3. **セル2** を実行 → ファイル選択ダイアログが出るので音声ファイルをアップロード
4. 文字起こしが終わると `.srt` ファイルが自動でダウンロードされます

> 音声ファイルのほか、動画ファイル(.mp4 / .mov)をそのままアップロードしてもOKです。

In [ ]:
#@title セル1: 準備(faster-whisperのインストール)
!pip install -q faster-whisper
print('✅ 準備完了。次のセルを実行してください。')

In [ ]:
#@title セル2: 音声をアップロード → 文字起こし → SRTをダウンロード
model_size = "large-v3"  #@param ["large-v3", "medium", "small"]

from google.colab import files
import ctranslate2, os, time

print('音声ファイル(または動画ファイル)を選択してください…')
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

device = 'cuda' if ctranslate2.get_cuda_device_count() > 0 else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'
if device == 'cpu':
    print('⚠️ GPUが無効です。CPUでも動きますが遅くなります(「ランタイムのタイプを変更」でGPUを推奨)')

from faster_whisper import WhisperModel
print(f'モデル {model_size} を読み込み中({device})…')
model = WhisperModel(model_size, device=device, compute_type=compute_type)

def srt_time(t):
    h = int(t // 3600); m = int(t % 3600 // 60); s = int(t % 60)
    ms = int(round((t - int(t)) * 1000))
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

print('文字起こしを開始します…')
start = time.time()
segments, info = model.transcribe(audio_path, language='ja', vad_filter=True)

blocks = []
for i, seg in enumerate(segments, 1):
    text = seg.text.strip()
    blocks.append(f'{i}\n{srt_time(seg.start)} --> {srt_time(seg.end)}\n{text}\n')
    print(f'[{srt_time(seg.start)} → {srt_time(seg.end)}] {text}')

srt_path = os.path.splitext(audio_path)[0] + '.srt'
with open(srt_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(blocks))

print(f'\n✅ 完了({time.time()-start:.0f}秒) → {srt_path} をダウンロードします')
files.download(srt_path)